# Zadanie 2 – Rekurencyjna metoda bisekcji

## Czego wymaga zadanie?

Zdefiniować rekurencyjną funkcję `bisection_rek(xlewy, xprawy, eps, f)`, która:  
- szuka **pierwiastka** funkcji `f` w przedziale `[xlewy, xprawy]`  
- zatrzymuje się gdy wynik jest dokładny do `eps`  
- zakłada, że `f(xlewy)` i `f(xprawy)` mają **różne znaki** (gwarantuje istnienie pierwiastka)

Przetestować dla funkcji:
$$f(x) = x^3 - 8x^2 - 35x + 150$$
w przedziałach $[-6, -4]$, $[0, 5]$, $[8, 11]$ z dokładnością $\varepsilon = 10^{-7}$

## Idea metody

Metoda bisekcji opiera się na **twierdzeniu Darboux (o wartości pośredniej)**:  
jeśli ciągła funkcja ma różne znaki na końcach przedziału, to gdzieś wewnątrz musi przejść przez zero.

Strategia: dzielimy przedział na pół, sprawdzamy w której połowie zmiana znaku, tam szukamy dalej.  
Każda iteracja **zmniejsza przedział o połowę** → precyzja rośnie wykładniczo.

## Implementacja

In [ ]:
def bisection_rek(xlewy, xprawy, eps, f):
    """
    Rekurencyjna metoda bisekcji.
    Szuka pierwiastka funkcji f w przedziale [xlewy, xprawy] z dokładnością eps.
    Założenie: f(xlewy) i f(xprawy) mają różne znaki.
    """
    # Wyznacz punkt środkowy przedziału
    xsrodek = (xlewy + xprawy) / 2

    # Warunek stopu: wystarczająca dokładność
    # |f(xsrodek)| < eps: wartość funkcji bliska zeru – to jest pierwiastek
    # |xprawy - xlewy| < eps: przedział tak mały, że dokładność wystarczająca
    if abs(f(xsrodek)) < eps or abs(xprawy - xlewy) < eps:
        return xsrodek

    # Sprawdź, w której połowie przedziału leży pierwiastek
    # Jeśli f(xlewy) i f(xsrodek) mają różne znaki → pierwiastek w lewej połowie
    if f(xlewy) * f(xsrodek) < 0:
        return bisection_rek(xlewy, xsrodek, eps, f)  # zawężamy przedział od prawej

    # W przeciwnym razie f(xsrodek) i f(xprawy) mają różne znaki → pierwiastek w prawej połowie
    else:
        return bisection_rek(xsrodek, xprawy, eps, f)  # zawężamy przedział od lewej

### Wizualizacja działania

```
Przedział [xlewy, xprawy], f(xlewy)<0, f(xprawy)>0:

  f < 0        |        f > 0
  xlewy -------+------- xprawy
            xsrodek
  
  Jeśli f(xsrodek) > 0: pierwiastek w [xlewy, xsrodek]
  Jeśli f(xsrodek) < 0: pierwiastek w [xsrodek, xprawy]
  Jeśli f(xsrodek) ≈ 0: ZNALEZIONO pierwiastek
```

Każde wywołanie rekurencyjne **zmniejsza długość przedziału o połowę**.

## Definicja funkcji testowej i testy

$$f(x) = x^3 - 8x^2 - 35x + 150$$

In [ ]:
def f(x):
    # Wielomian z treści zadania
    return x**3 - 8*x**2 - 35*x + 150

In [ ]:
# Weryfikacja, że na końcach każdego przedziału f ma różne znaki
# (wymagane przez algorytm – inaczej nie ma gwarancji istnienia pierwiastka)
przedzialy = [(-6, -4), (0, 5), (8, 11)]

print("Sprawdzenie znaków funkcji na końcach przedziałów:")
print(f"{'Przedział':>12}  {'f(xlewy)':>12}  {'f(xprawy)':>12}  {'Różne znaki?':>14}")
print("-" * 56)
for xlewy, xprawy in przedzialy:
    fl = f(xlewy)
    fp = f(xprawy)
    rozne = "TAK" if fl * fp < 0 else "NIE – UWAGA!"
    print(f"[{xlewy:>3}, {xprawy:>3}]  {fl:>12.4f}  {fp:>12.4f}  {rozne:>14}")

In [ ]:
eps = 1e-7

print(f"Szukanie pierwiastków z dokładnością eps = {eps}\n")
print(f"{'Przedział':>12}  {'Pierwiastek':>15}  {'f(pierwiastek)':>16}")
print("-" * 48)
for xlewy, xprawy in przedzialy:
    x0 = bisection_rek(xlewy, xprawy, eps, f)
    print(f"[{xlewy:>3}, {xprawy:>3}]  {x0:>15.8f}  {f(x0):>16.2e}")

## Weryfikacja wyników

Rozkładamy wielomian $f(x) = x^3 - 8x^2 - 35x + 150$ na czynniki, by znaleźć dokładne pierwiastki.

In [ ]:
import numpy as np

# Współczynniki wielomianu: x^3 - 8x^2 - 35x + 150
wspolczynniki = [1, -8, -35, 150]
pierwiastki_dokladne = sorted(np.roots(wspolczynniki))

print("Dokładne pierwiastki (numpy.roots):")
for p in pierwiastki_dokladne:
    print(f"  x = {p.real:.8f}  (f(x) = {f(p.real):.2e})")

print()
print("Wyniki bisekcji:")
for xlewy, xprawy in przedzialy:
    x0 = bisection_rek(xlewy, xprawy, eps, f)
    print(f"  Przedział [{xlewy}, {xprawy}]: x = {x0:.8f}")

## Liczba iteracji i zbieżność

In [ ]:
# Wersja śledząca każdy krok – pokazuje zawężanie przedziału
kroki = []

def bisection_z_krokami(xlewy, xprawy, eps, f, krok=0):
    xsrodek = (xlewy + xprawy) / 2
    kroki.append((krok, xlewy, xprawy, xsrodek, f(xsrodek)))
    if abs(f(xsrodek)) < eps or abs(xprawy - xlewy) < eps:
        return xsrodek
    if f(xlewy) * f(xsrodek) < 0:
        return bisection_z_krokami(xlewy, xsrodek, eps, f, krok + 1)
    else:
        return bisection_z_krokami(xsrodek, xprawy, eps, f, krok + 1)

kroki.clear()
wynik = bisection_z_krokami(0, 5, eps, f)

print(f"Bisekcja na [0, 5], eps={eps}")
print(f"{'Krok':>5} {'xlewy':>12} {'xprawy':>12} {'xsrodek':>12} {'f(xsrodek)':>14}")
print("-" * 60)
for krok, xl, xp, xs, fxs in kroki[:25]:  # pierwsze 25 kroków
    print(f"{krok:>5} {xl:>12.6f} {xp:>12.6f} {xs:>12.6f} {fxs:>14.6e}")
print(f"\nŁączna liczba kroków: {len(kroki)}, wynik: {wynik:.8f}")

## Podsumowanie

| Element | Opis |
|---|---|
| **Funkcja** | `bisection_rek(xlewy, xprawy, eps, f)` |
| **Warunek stopu** | `\|f(xsrodek)\| < eps` lub `\|xprawy-xlewy\| < eps` |
| **Rekurencja lewa** | `f(xlewy) * f(xsrodek) < 0` → pierwiastek w lewej połowie |
| **Rekurencja prawa** | w przeciwnym razie → pierwiastek w prawej połowie |
| **Wymaganie** | `f(xlewy)` i `f(xprawy)` muszą mieć różne znaki |
| **Złożoność** | $O(\log_2((x_{prawy}-x_{lewy})/\varepsilon))$ kroków |